# The p-Center Problem: Minimax Facility Location

**Problem.** Given $n$ points and the distance between every ordered pair, choose exactly $p$ of them to act as centers and assign every remaining point to one center, so that the largest point-to-center distance is as small as possible. This minimax objective is the right one whenever the binding requirement is a guarantee rather than an average: ambulance and fire station siting, cell tower placement, disaster relief depots, or field service territories with a maximum response time. It contrasts with the $p$-median problem, which minimizes total distance and will accept one
badly served outlier in exchange for a better mean.

Three instances are solved: two to proven optimality, and one large instance under a time limit. The final section measures how the algebraic form of a single constraint changes the strength of the linear relaxation.

## Formulation

**Sets.** Points $i, j \in V = \{1, \ldots, n\}$. Every point is both a demand point and a candidate center.

**Parameters.**

| Symbol | Meaning |
| --- | --- |
| $d_{ij}$ | Distance from point $i$ to point $j$ |
| $p$ | Number of centers to open |

**Decision variables.**

- $y_j \in \{0,1\}$, 1 if point $j$ is opened as a center.
- $x_{ij} \in \{0,1\}$, 1 if non-center point $i$ is assigned to center $j$, for $i \neq j$.
- $\eta \ge 0$, the covering radius, meaning the largest realized assignment distance.


**Model.**

$$
\begin{aligned}
\min \quad & \eta \\
\text{s.t.}\quad
& \sum_{j \in V} y_j = p && && \text{(1) open exactly } p\text{ centers}\\
& y_i + \sum_{j \neq i} x_{ij} = 1 && \forall\, i \in V && \text{(2) point is either center or assigned to one}\\
& x_{ij} \le y_j && \forall\, i \neq j && \text{(3) linking to open centers}\\
& d_{ij}\, x_{ij} \le \eta && \forall\, i \neq j && \text{(4) } \eta \text{ dominates every realized distance}\\
& x_{ij}, y_j \in \{0,1\}, \quad \eta \ge 0.
\end{aligned}
$$

**Linearising the minimax objective.** If stated directly, the problem is a nested $\min \max \min d_{ij}$, which is not linear. 
The constraint (4) removes the outer $\max$ by bounding every individual assignment distance by $\eta$; since the objective pushes $\eta$ down, at optimality it settles exactly on the largest one. The inner $\min$ is removed by constraint (2): each point is assigned to exactly one center, and because a shorter assignment always relaxes the constraint (4), the optimal solution never assigns a point to anything other than its nearest open center. Both nested operators become linear constraints, leaving a binary program with a single continuous variable.

The constraint (2) is an equality that includes $y_i$, which encodes the convention that a point which is itself a center is not assigned to anything and contributes no distance to $\eta$. The objective therefore measures distance from non-centers only.

**An equivalent $\eta$ constraint.** Because the constraint (2) forces $\sum_{j \neq i} x_{ij} = 1$ for every non-center, the per-pair constraints (4) can be collapsed into one constraint per point:

$$\sum_{j \neq i} d_{ij}\, x_{ij} \le \eta \qquad \forall\, i \in V \qquad \text{(4a) aggregated realized distance}$$

Over the integers the two are equivalent, since exactly one term in the sum is nonzero. Over the reals they are not, and Section 6 measures the difference. Both are implemented and selected by the `Aggregated` flag on `build_model`; everything before Section 6 uses the Disaggregated form.

## 1. Data

Each instance file gives $n$, $p$, and the full $n \times n$ distance matrix. The files are read exactly as supplied, with no edits to the source.

In [1]:
import re
from pathlib import Path

import numpy as np
import gurobipy as gp
from gurobipy import GRB

DATA_DIR = Path(".")
INSTANCES = ["PC001.dat", "PC002.dat", "PC003.dat"]

In [2]:
def read_instance(path):
    """Parse a p-center instance file into (n, p, D)."""
    text = Path(path).read_text()

    n = int(re.search(r"N:\s*(\d+)", text).group(1))
    p = int(re.search(r"P:\s*(\d+)", text).group(1))

    block = text.split("DIST:", 1)[1]
    block = block[block.index("[") + 1 : block.rindex("]")]
    values = [int(v) for v in block.split()]

    if len(values) != n * n:
        raise ValueError(f"{path}: expected {n * n} distances, parsed {len(values)}")

    return n, p, np.array(values, dtype=int).reshape(n, n)

In [3]:
print(f"{'Instance':<12}{'Points':>8}{'Centers':>9}{'Max distance':>14}"
      f"{'Variables':>11}{'Rows':>9}")
print("-" * 63)
for name in INSTANCES:
    n, p, D = read_instance(DATA_DIR / name)
    n_var = 1 + n + n * (n - 1)
    n_row = 1 + n + 2 * n * (n - 1)
    print(f"{name:<12}{n:>8}{p:>9}{D.max():>14}{n_var:>11,}{n_row:>9,}")

Instance      Points  Centers  Max distance  Variables     Rows
---------------------------------------------------------------
PC001.dat         10        3           981        101      191
PC002.dat        100       15          9999     10,001   19,901
PC003.dat        200       25         20000     40,001   79,801


## 2. Model

One builder covers both the constraint (4) and (4a) formulations. `report` prints a solution and cross-checks it against
the raw distance matrix, and `progress_report` summarizes a recorded search trajectory.

In [4]:
def build_model(n, p, D, Aggregated=False, name="p-center"):
    """Construct the p-center integer program.

    `Aggregated` selects the \eta constraint (4a) form: 
    False for the per-pair version, True for the one-per-point version. 
    The two are equivalent over the integers and differ sharply in the LP relaxation (Section 6).
    """
    model = gp.Model(name)

    eta = model.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="radius")
    y = model.addVars(n, vtype=GRB.BINARY, name="open")
    x = model.addVars([(i, j) for i in range(n) for j in range(n) if i != j], vtype=GRB.BINARY, name="assign")

    model.setObjective(eta, GRB.MINIMIZE)

    model.addConstr(y.sum() == p, name="centers")

    model.addConstrs(
        (y[i] + gp.quicksum(x[i, j] for j in range(n) if j != i) == 1 for i in range(n)), name="assignment")

    model.addConstrs((x[i, j] <= y[j] for i in range(n) for j in range(n) if i != j), name="linking")

    if Aggregated:
        model.addConstrs(
            (gp.quicksum(float(D[i, j]) * x[i, j] for j in range(n) if j != i) <= eta for i in range(n)), 
            name="radius")
    else:
        model.addConstrs(
            (float(D[i, j]) * x[i, j] <= eta for i in range(n) for j in range(n) if i != j),  name="radius")

    model.update()
    return model, x, y, eta

In [5]:
def extract_solution(n, x, y):
    """Pull the center set and the assignment map out of a solved model."""
    centers = sorted(j + 1 for j in range(n) if y[j].X > 0.5)
    assignment = {(i + 1): (j + 1) for i in range(n) for j in range(n) if i != j and x[i, j].X > 0.5}
    return centers, assignment


def report(name, n, p, D, centers, assignment, radius, runtime, status):
    """Print a solution report, having first cross-checked the radius.
    A solver reporting OPTIMAL confirms the model was solved correctly.
    """
    assert max(D[i - 1, j - 1] for i, j in assignment.items()) == radius

    print(f"{'=' * 58}\n{name}   (n = {n}, p = {p})\n{'=' * 58}")
    print(f"  Status    : {status}")
    print(f"  Radius    : {radius:.2f}")
    print(f"  Runtime   : {runtime:.2f} s")
    print(f"  Centers   : {centers}")

In [6]:
def track(model, where):
    """Record (runtime, incumbent, bound) throughout branch-and-bound.
    This captures the trajectory while the search is running.
    """
    if where == GRB.Callback.MIP:
        best = model.cbGet(GRB.Callback.MIP_OBJBST)
        if best < GRB.INFINITY:
            history.append((model.cbGet(GRB.Callback.RUNTIME), best,
                            model.cbGet(GRB.Callback.MIP_OBJBND)))


def progress_report(history, total, optimum, gap_threshold=0.10):
    """Print when the search reached each milestone, as a fraction of total runtime."""
    first = lambda ok: next((t for t, best, bound in history if ok(best, bound)), None)

    milestones = [
        ("First feasible solution found", first(lambda b, bd: True)),
        ("Optimal solution first found", first(lambda b, bd: b <= optimum + 1e-6)),
        (f"Gap first below {gap_threshold:.0%}",
         first(lambda b, bd: b != 0 and abs(b - bd) / abs(b) < gap_threshold)) ]

    print(f"{'Milestone':<34}{'Time':>10}{'Fraction':>12}")
    print("-" * 56)
    for label, t in milestones:
        cell = f"{t:>9.2f}s{t / total:>11.1%}" if t is not None else f"{'n/a':>10}{'n/a':>12}"
        print(f"{label:<34}{cell}")

## 3. PC001: 10 points, 3 centers

Only $n(n-1) = 90$ assignment variables, one per ordered pair. Small enough to solve at the root node.

In [7]:
n, p, D = read_instance(DATA_DIR / "PC001.dat")

model, x, y, eta = build_model(n, p, D)
model.optimize()

centers, assignment = extract_solution(n, x, y)
report("PC001.dat", n, p, D, centers, assignment, model.ObjVal, model.Runtime,
       "Optimal" if model.Status == GRB.OPTIMAL else str(model.Status))

print("\n  Assignments:")
for i in sorted(assignment):
    d = D[i - 1, assignment[i] - 1]
    flag = "   <-- binding" if d == model.ObjVal else ""
    print(f"    point {i:>3}  ->  center {assignment[i]:>3}   (d = {d}){flag}")

Set parameter Username
Academic license - for non-commercial use only - expires 2027-02-13
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i7-10510U CPU @ 1.80GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 191 rows, 101 columns and 470 nonzeros (Min)
Model fingerprint: 0x99932cbd
Model has 1 linear objective coefficients
Variable types: 1 continuous, 100 integer (100 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+03]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 3e+00]

Found heuristic solution: objective 810.0000000
Presolve removed 94 rows and 14 columns
Presolve time: 0.01s
Presolved: 97 rows, 87 columns, 334 nonzeros
Variable types: 0 continuous, 87 integer (86 binary)

Root relaxation: objective 8.855732e+01, 44 iterations, 0.00 seconds (0.00 work units)

    Nodes  

Almost every assignment sits well below the radius, and a single pair determines the objective. This is the defining behavior of a minimax model.

## 4. PC002: 100 points, 15 centers

The assignment variables number $n(n-1) = 100 \times 99 = 9{,}900$, one for each ordered pair $i \neq j$, plus 100 location variables and the radius. A callback records the incumbent and the bound throughout the search, which turns "how long did it take" into "where did the time actually go".

In [8]:
n, p, D = read_instance(DATA_DIR / "PC002.dat")
history = []

model, x, y, eta = build_model(n, p, D)
model.optimize(track)

centers, assignment = extract_solution(n, x, y)
report("PC002.dat", n, p, D, centers, assignment, model.ObjVal, model.Runtime,
       "Optimal" if model.Status == GRB.OPTIMAL else str(model.Status))

print()
progress_report(history, model.Runtime, model.ObjVal)

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i7-10510U CPU @ 1.80GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 19901 rows, 10001 columns and 49700 nonzeros (Min)
Model fingerprint: 0xe9a9681e
Model has 1 linear objective coefficients
Variable types: 1 continuous, 10000 integer (10000 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+04]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+01]

Found heuristic solution: objective 9919.0000000
Presolve removed 9813 rows and 13 columns
Presolve time: 0.49s
Presolved: 10088 rows, 9988 columns, 39848 nonzeros
Variable types: 0 continuous, 9988 integer (9987 binary)

Root relaxation: objective 2.846574e+02, 1567 iterations, 0.28 seconds (0.29 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Ex

Almost the entire solve is search, not proof. A feasible solution appears at 2.2% of the runtime, the eventual optimum is not found until 92.6%, and the proof that follows takes 1.75s.

The gap crossing 10% at that same instant is no coincidence. The bound had already reached 621 while the incumbent was still 702, so finding 684 closed the gap to 9.21% on its own. The bound was never the obstacle here; finding the right solution was.

That limits what an early stop buys. Stopping at 15s returns 702, only 2.6% above optimal, but stopping at a 10% gap saves nothing, because the run reaches that threshold only by finding the optimum.

## 5. PC003: 200 points, 25 centers

Now $200 \times 199 = 39{,}800$ assignment variables, and no realistic prospect of proving optimality. This one runs under a ten minute limit and reports the best solution found together with the gap that remains, which for a model of this size is the realistic operating mode rather than a
fallback.

In [9]:
TIME_LIMIT = 600   # seconds

n, p, D = read_instance(DATA_DIR / "PC003.dat")
history = []

model, x, y, eta = build_model(n, p, D)
model.Params.TimeLimit = TIME_LIMIT
model.optimize(track)

status = {GRB.OPTIMAL: "Optimal", 
          GRB.TIME_LIMIT: "Time limit reached"}.get(model.Status, str(model.Status))

centers, assignment = extract_solution(n, x, y)
report("PC003.dat", n, p, D, centers, assignment, model.ObjVal, model.Runtime, status)

print(f"  Bound     : {model.ObjBound:.2f}")
print(f"  Gap       : {model.MIPGap:.2%}")

t_first = next((t for t, best, bound in history), None)
if t_first is not None:
    print(f"\n  First feasible solution at {t_first:.2f} s " 
          f"({t_first / model.Runtime:.1%} of the limit)")

Set parameter TimeLimit to value 600
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i7-10510U CPU @ 1.80GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  600

Optimize a model with 79801 rows, 40001 columns and 199400 nonzeros (Min)
Model fingerprint: 0x2d0bb675
Model has 1 linear objective coefficients
Variable types: 1 continuous, 40000 integer (40000 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+04]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 3e+01]

Found heuristic solution: objective 19855.000000
Presolve removed 39622 rows and 22 columns
Presolve time: 1.75s
Presolved: 40179 rows, 39979 columns, 159712 nonzeros
Variable types: 0 continuous, 39979 integer (39978 binary)
Deterministic concurrent LP optimizer: primal and dual simplex
Showing primal log on

Ten minutes is not close to enough. The run ends with a best known radius of 1085 against a bound of 795, a gap of 26.7%, and the bound gained about one unit over the final twenty seconds.

A feasible solution appears after 1.84s, but the rest of the run is grinding: 14,811 nodes and 9.1 million simplex iterations. The last burst comes at 579s, where the incumbent falls from 1092 to 1085 before the limit cuts the search off mid-descent.

This is a best known radius, not a proven one. The assertion in `report` confirms 1085 is what this center set achieves, which is a separate question from whether a better one exists.

## 6. Formulation strength

Written pair by pair, the constraint (4)

$$d_{ij}\, x_{ij} \le \eta \qquad \forall\, i \neq j$$

relaxes badly. A point can spread a fractional assignment across many centers, and because each constraint sees a single pair, the relaxation never charges more than the largest term $d_{ij} x_{ij}$, which any small fraction makes cheap.

Since the constraint (2) forces $\sum_{j \neq i} x_{ij} = 1$, the same condition can be written once per point as the constraint (4a):

$$\sum_{j \neq i} d_{ij}\, x_{ij} \le \eta$$

A fractional assignment now pays a weighted average rather than hiding behind its cheapest component. The integer feasible set is unchanged, and $n^2 - n$ rows become $n$.

In [10]:
print(f"{'Instance':<12}{'Form':<16}{'Rows':>9}{'LP bound':>11}{'Improvement':>13}")
print("-" * 61)

for name in INSTANCES:
    n, p, D = read_instance(DATA_DIR / name)
    bounds = {}
    for label, flag in [("Disaggregated", False), ("Aggregated", True)]:
        m, *_ = build_model(n, p, D, Aggregated=flag)
        m.Params.OutputFlag = 0
        lp = m.relax()
        lp.Params.OutputFlag = 0
        lp.optimize()
        bounds[label] = lp.ObjVal
        gain = (f"{bounds['Aggregated'] / bounds['Disaggregated']:.1f}x"
                if flag and bounds["Disaggregated"] > 0 else "")
        print(f"{name if not flag else '':<12}{label:<16}{m.NumConstrs:>9,}"
              f"{lp.ObjVal:>11.2f}{gain:>13}")
    print()

Instance    Form                 Rows   LP bound  Improvement
-------------------------------------------------------------
PC001.dat   Disaggregated         191      22.54             
            Aggregated            111      88.56         3.9x

PC002.dat   Disaggregated      19,901      17.18             
            Aggregated         10,101     284.66        16.6x

PC003.dat   Disaggregated      79,801      16.04             
            Aggregated         40,201     356.98        22.3x



Aggregating is worth 3.9x to 22.3x on the root bound, and almost halves the row count.

Gurobi largely finds this on its own, though. The PC002 log in Section 4 reports a root relaxation of 284.66, the aggregated bound, not 17.18. Presolve cut 19,901 rows to 10,088, almost exactly the 10,101 the aggregated form states outright, and PC001 and PC003 behave the same way. So writing it aggregated buys a smaller model, not a bound the solver would have missed.

What neither form fixes is the gap that remains. Even 284.66 sits far below 684 on PC002, and 356.98 below the 1085 on PC003. That distance is intrinsic to the minimax structure, and it is why PC003 is still 26.7 percent open.

## Conclusion

| Instance | $n$ | $p$ | Radius | Status | Runtime |
| --- | ---: | ---: | ---: | --- | ---: |
| `PC001.dat` | 10 | 3 | **257** | Optimal | 0.07 s |
| `PC002.dat` | 100 | 15 | **684** | Optimal | 23.59 s |
| `PC003.dat` | 200 | 25 | **1085** (best found) | Time limit, gap 26.73% | 600.08 s |

Every radius is recomputed from the raw distance matrix and asserted against the solver objective, so none of them rests on Gurobi's own output alone.

- **Difficulty scales badly.** Doubling $n$ from 100 to 200 turns a 24 second proof into ten minutes that still leave a quarter of the gap open. The model is easy to write and hard to solve.
- **The search finds answers faster than it can justify them.** On PC002 the optimum arrives at 92.6 percent of the runtime; on PC003 a good solution arrives in under two seconds and is never proven. Where a certificate is not needed, most of the time spent buys nothing.
- **A better formulation would help more than a better solver.** Aggregating the constraint (4) lifts the LP bound up to 22 times, but Gurobi's presolve already does that, and even the aggregated bound sits far below the optimum. Getting past PC003 means the set-covering reformulation with binary search over the distinct distances, not more time on this one.